In [ ]:
import joblib
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)


class ApplyKmeans:
    def __init__(self, device):
        self.km_model = joblib.load(
            "/home/alex/Projekt/cfm-vc/ckpt/xeus/kmeans_xeus.pkl"
        )
        self.C_np = self.km_model.cluster_centers_.transpose()
        self.Cnorm_np = (self.C_np**2).sum(0, keepdims=True)
        self.device = device

        self.C = torch.from_numpy(self.C_np).to(device)
        self.Cnorm = torch.from_numpy(self.Cnorm_np).to(device)

    def __call__(self, x):
        x = x.to(self.device)
        dist = x.pow(2).sum(1, keepdim=True) - 2 * torch.matmul(x, self.C) + self.Cnorm
        return dist.argmin(dim=1).cpu().numpy()


apply_kmeans = ApplyKmeans(device)

In [ ]:
from espnet2.tasks.ssl import SSLTask
import librosa
import torch
from torchaudio.transforms import Resample
import librosa

resampler = Resample(orig_freq=24000, new_freq=16000).to(device)

xeus_model, xeus_train_args = SSLTask.build_model_from_file(
    "/home/alex/Projekt/cfm-vc/ckpt/xeus/config.yaml",
    "/home/alex/Projekt/cfm-vc/ckpt/xeus/xeus_checkpoint_new.pth",
    device,
)

In [ ]:
# INFO_TPL_1441_FIRSTWARN_CONDITION_LESTER_15_01.mp3
# INFO_CORANGAR_FINDHERB_SUCCESS_08_02.mp3
wavs, sampling_rate = librosa.load(
    "/home/alex/Data/gothic 3/deu/svm_player_bookofzuben_06_german.mp3"
)
wavs = torch.from_numpy(wavs).float().unsqueeze(0).to(device)  # (1, time)
wavs_16k = resampler(wavs)
wavs_16k_lengths = torch.LongTensor([wavs_16k.shape[1]]).to(device)

with torch.no_grad():
    final_output, hidden_states, feat_lengths = xeus_model.inference_encode(
        wavs_16k, wavs_16k_lengths, use_mask=False
    )

features = hidden_states[14].squeeze(0)
features.shape

In [ ]:
units = apply_kmeans(features).tolist()
units

In [ ]:
import torch
import torch.nn as nn
import torchaudio

# plot the mel spectrogram
import matplotlib.pyplot as plt


def safe_log(x: torch.Tensor, clip_val: float = 1e-7) -> torch.Tensor:
    """
    Computes the element-wise logarithm of the input tensor with clipping to avoid near-zero values.

    Args:
        x (Tensor): Input tensor.
        clip_val (float, optional): Minimum value to clip the input tensor. Defaults to 1e-7.

    Returns:
        Tensor: Element-wise logarithm of the input tensor with clipping applied.
    """
    return torch.log(torch.clip(x, min=clip_val))


class MelSpectrogramFeatures(nn.Module):
    def __init__(
        self,
        sample_rate=24000,
        n_fft=1024,
        hop_length=256,
        n_mels=100,
        padding="center",
    ):
        super().__init__()
        if padding not in ["center", "same"]:
            raise ValueError("Padding must be 'center' or 'same'.")
        self.padding = padding
        self.mel_spec = torchaudio.transforms.MelSpectrogram(
            sample_rate=sample_rate,
            n_fft=n_fft,
            hop_length=hop_length,
            n_mels=n_mels,
            center=padding == "center",
            power=1,
        )

    def forward(self, audio, **kwargs):
        if self.padding == "same":
            pad = self.mel_spec.win_length - self.mel_spec.hop_length
            audio = torch.nn.functional.pad(audio, (pad // 2, pad // 2), mode="reflect")
        mel = self.mel_spec(audio)
        features = safe_log(mel)
        return features


mel_extractor = MelSpectrogramFeatures(
    sample_rate=24000, n_fft=1024, hop_length=480, n_mels=100, padding="same"
).to(device)

mel_features = mel_extractor(wavs)
print(mel_features.shape)

plt.figure(figsize=(10, 4))
plt.imshow(mel_features.squeeze(0).cpu().numpy(), aspect="auto", origin="lower")
plt.title("Mel Spectrogram")
plt.xlabel("Time")
plt.ylabel("Mel Frequency")
plt.colorbar(format="%+2.0f dB")
plt.tight_layout()
plt.show()

In [ ]:
from glob import glob

audio_files = glob("/home/alex/Data/gothic 3/deu/*.mp3")
print(f"Found {len(audio_files)} audio files.")


diffs = 0

for audio_file in audio_files:
    wavs, sampling_rate = librosa.load(audio_file)
    wavs = torch.from_numpy(wavs).float().unsqueeze(0).to(device)  # (1, time)
    wavs_16k = resampler(wavs)
    wavs_16k_lengths = torch.LongTensor([wavs_16k.shape[1]]).to(device)

    with torch.no_grad():
        final_output, hidden_states, feat_lengths = xeus_model.inference_encode(
            wavs_16k, wavs_16k_lengths, use_mask=False
        )

    features = hidden_states[14].squeeze(0)

    units = apply_kmeans(features).tolist()

    mel_features = mel_extractor(wavs)

    mel_features = mel_features[:, :, : len(units)]

    if mel_features.shape[2] != len(units):
        diffs += 1
        print(f"File: {audio_file} - Mel frames: {mel_features.shape[2]}, Units: {len(units)}")



In [ ]:
import torch
import torch.nn as nn
import librosa
import torchaudio

device = "cuda" if torch.cuda.is_available() else "cpu"

wavs, sampling_rate = librosa.load(
    "/home/alex/Data/gothic 3/deu/svm_player_bookofzuben_06_german.mp3", sr=24000
)
wavs = torch.from_numpy(wavs).float().unsqueeze(0).to(device)  # (1, time)


def safe_log(x: torch.Tensor, clip_val: float = 1e-7) -> torch.Tensor:
    """
    Computes the element-wise logarithm of the input tensor with clipping to avoid near-zero values.

    Args:
        x (Tensor): Input tensor.
        clip_val (float, optional): Minimum value to clip the input tensor. Defaults to 1e-7.

    Returns:
        Tensor: Element-wise logarithm of the input tensor with clipping applied.
    """
    return torch.log(torch.clip(x, min=clip_val))


class MelSpectrogramFeatures(nn.Module):
    def __init__(
        self,
        sample_rate=24000,
        n_fft=1024,
        hop_length=256,
        n_mels=100,
        padding="center",
    ):
        super().__init__()
        if padding not in ["center", "same"]:
            raise ValueError("Padding must be 'center' or 'same'.")
        self.padding = padding
        self.mel_spec = torchaudio.transforms.MelSpectrogram(
            sample_rate=sample_rate,
            n_fft=n_fft,
            hop_length=hop_length,
            n_mels=n_mels,
            center=padding == "center",
            power=1,
        )

    def forward(self, audio, **kwargs):
        if self.padding == "same":
            pad = self.mel_spec.win_length - self.mel_spec.hop_length
            audio = torch.nn.functional.pad(audio, (pad // 2, pad // 2), mode="reflect")
        mel = self.mel_spec(audio)
        features = safe_log(mel)
        return features


mel_extractor = MelSpectrogramFeatures(
    sample_rate=24000, n_fft=1024, hop_length=256, n_mels=100, padding="same"
).to(device)

mel_features = mel_extractor(wavs)
print(mel_features.shape)


In [ ]:
import torch

from vocos import Vocos

vocos = Vocos.from_pretrained("charactr/vocos-mel-24khz").cuda()

audio = vocos.decode(mel_features)

In [ ]:
# play audio
import IPython.display as ipd
ipd.Audio(audio.cpu().numpy(), rate=24000)

In [ ]:
"INFO_CORANGAR_FINDHERB_SUCCESS_08_02.mp3".lower()

In [ ]:
import torch
import numpy as np

mel_np = np.load("/home/alex/Projekt/cfm-vc/outputs/smoke_test.npy")

mel = torch.from_numpy(mel_np).to(device)

audio = vocos.decode(mel.unsqueeze(0))

ipd.Audio(audio.cpu().numpy(), rate=24000)